# Running the Conditionally Gaussian Kalman Filter (CGKF) on Lorenz-63

This notebook shows the minimal pattern for running `CGKF` on a conditionally Gaussian system: define the model's drift functions `f_uI` (observed state) and `f_uII` (hidden state), load/generate a truth trajectory, and call `CGKF` directly. The filter derives its own linearization (the A0, A1, B0, B1 matrices) internally via finite differences -- you do **not** need to hand-derive them yourself.

**Setup** (once, from the repo root):
```bash
pip install -e .
```
This notebook assumes it is run with the repo root (`data_assimilation/`) as the working directory, so that the `da` package is importable and the results save path resolves correctly.

The settings below (`R0`, `Sigma`, `Gamma`) match the validated configuration used to produce the Lorenz-63 results in the paper (`scripts/run_l63_comparison.py`).

In [ ]:
import numpy as np
from da.models import lorenz63
from da.filters import CGKF
from da.data import load_lorenz63
from da.run_diagnostics import run_l63_dashboard

In [3]:
true_obj = load_lorenz63()

## 1. Load the truth trajectory

`load_lorenz63()` generates a Lorenz-63 trajectory on the fly (Euler-Maruyama integration) -- no external data file is needed. `x` is treated as the observed state; `(y, z)` are the hidden states the filter will recover.

In [4]:
t_vals = true_obj.t

## 2. Run the filter

`f_uI` and `f_uII` (from `lorenz63()`) fully define the model: `x` is the observed state, `(y, z)` the hidden state. Passing them directly to `CGKF` (with no `precomputed_matrices`) lets the filter derive its own linearization internally -- this is the simplest way to run CGKF on a new system, and avoids hand-deriving/transcribing the A0, A1, B0, B1 matrices yourself (a common source of bugs).

`R0`, `Sigma`, and `Gamma` are the initial hidden-state covariance, process noise, and observation noise, matching the paper's validated setup.

In [ ]:
f_uI, f_uII, params = lorenz63()
np.random.seed(0)
uII_0 = true_obj.mu[0] + np.random.randn(2)

pred_obj = CGKF(
    d_uI=f_uI,
    d_uII=f_uII,
    t_span=(true_obj.t[0], true_obj.t[-1]),
    uI=true_obj.U,
    uII_0=uII_0,
    R0=1 * np.eye(2),
    Sigma=0.1 * np.eye(2),
    Gamma=np.array([[0.5]]),
    method="euler",
)

In [6]:
print(np.sum(np.isnan(pred_obj.mu[:,1]) == 1))  # Check for NaNs in the predicted mean
print(np.isnan(pred_obj.cov).any())  # Check for NaNs in the predicted covariance

0
False


In [ ]:
run_l63_dashboard(true_obj, pred_obj, save_dir='results/CGKF/lorenz63')